# FORESEE Models: ALP coupling to Fermions

## Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
import src.foresee as foresee_module
from matplotlib import pyplot as plt

## 1. Specifying the Model

The phenomenology of an ALP with universal fermion couplings can be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = - \frac{1}{2} \textcolor{red}{m_{a}}^2 a^2  + \frac{\partial_\mu a}{\textcolor{red}{f_a}} \left(\sum_{\ell} c_\ell(\mu)\bar{\ell} \gamma^\mu\gamma_5 \ell +  \sum_{q}c_q(\mu)\bar{q} \gamma^\mu\gamma_5 q  + \sum_{i\neq j}c_{ij}(\mu)\bar{d_i} \gamma^\mu\gamma_5 d_j \right)
\end{equation}
where the ALP mass $\textcolor{red}{m_a}$, the suppression scale $\textcolor{red}{f_a}$, and the scale $\Lambda$ where the universal coupling is defined $c_i(\Lambda) = 1$ are free parameters. The coefficients $c_q,c_l, c_{ij}$ are scale-dependent parameters which arise from the RG evolution of the flavour-diagonal couplings from $\Lambda$ to the energy scale of the process in consideration, $\mu$.  For the search for ALPs at forward experiments we need to know i) the *production rate*, and ii) the *interaction rate*. All these properties are specified in the `Model` class. We initialize it with the name of the model as argument. 

In [ ]:
energy = "14"
modelname="ALP-fermion"
model = Model(modelname, path="./")

# Builder parameters, matching the build.py / load_model() defaults.
nsample_2body = 2000
generators_light = ["EPOSLHC", "SIBYLL", "QGSJET"][:1]
generators_heavy = ["NLO-P8", "NLO-P8-Max", "NLO-P8-Min"][:1]

**Production** The ALP is mainly produced in FCNC B-meson decays. Following [2201.05170](https://arxiv.org/pdf/2201.05170.pdf), the branching fractions are 

\begin{equation}
\text{BR}(B \to X_s a)     = \frac{1}{\Gamma_B} \frac{m_B^2 - m_a^2}{32\pi m_B^3}\left|\frac{c_{ij} m_b g_{aff}}{4v}\right|^2 
\end{equation}
where $c_{bs} = 0.0008$, $m_B$ is the mass of the B-meson, and $m_b$ is the b-quark mass. 

In the following, we model light hadron production using `EPOSLHC`, `SIBYLL`, `QGSJET` and `Pythia8-Forward` and heavy hadron production using the `POWHEG+Pythia8` predicions.

In [ ]:
for pid0 in ['511', '521', '531', '541']: 
    model.add_production_2bodydecay(
        pid0 = pid0,
        pid1 = '3',
        br = "(1.8e-3*coupling/(2*246.22))**2*(self.masses('pid0')**2 - mass**2)/(32*np.pi*self.masses('pid0')**3*self.widths('pid0'))",
        generator = generators_heavy,
        energy = energy,
        nsample = nsample_2body, 
    ) 

    model.add_production_2bodydecay(
        pid0 = f"-{pid0}",
        pid1 = '-3',
        br = "(1.8e-3*coupling/(2*246.22))**2*(self.masses('pid0')**2 - mass**2)/(32*np.pi*self.masses('pid0')**3*self.widths('pid0'))",
        generator = generators_heavy,
        energy = energy,
        nsample = nsample_2body, 
    ) 


We can also produce the ALP via its resonant mixing with the pseudo-scalar mesons, in particular the $\pi^0$, $\eta$ and $\eta '$ mesons. Following [2201.05170](https://arxiv.org/pdf/2201.05170.pdf), we can write 

\begin{equation}
\sigma(A') = \theta_P^2 \  \sigma(P)
\end{equation}


\begin{equation}\theta_{\pi^0} \approx \frac{f_\pi F_{VMD}(m_a)}{f} \frac{m_a^2}{(m_a^2 - m_{\pi^0}^2)} \left( (c_d - c_u) + \delta \frac{m_{\pi^0}^2}{3} \left[ \frac{m_a^2 (c_d + 2c_s + c_u)}{m_a^2 - m_{\eta'}^2} + \frac{2m_a^2 (-c_s + c_u + c_d)}{m_a^2 - m_\eta^2} \right] \right)
\end{equation}


\begin{equation}
\theta_{\eta} \approx \frac{f_\pi F_{VMD}(m_a)}{f} \sqrt{\frac{2}{3}} \frac{m_a^2}{m_a^2 - m_\eta^2} \left( (c_u + c_d - c_s) - \delta \frac{m_{\pi^0}^2 (c_u - c_d)}{m_a^2 - m_{\pi^0}^2} \right)
\end{equation}

\begin{equation}
\theta_{\eta'} \approx \frac{f_\pi F_{VMD}(m_a)}{f} \frac{1}{\sqrt{3}} \frac{m_a^2}{m_a^2 - m_{\eta'}^2} \left( -(c_d + 2c_s + c_u) - \delta \frac{m_{\pi^0}^2 (c_u - c_d)}{m_{\pi^0}^2 - m_a^2} \right)
\end{equation}

where $c_d, c_u, c_s$ are coefficients which depend on the RG flow from $\Lambda = 1 \text{ TeV}$ down to $Q\sim \text{GeV}$, and $F_{VMD}$ is the form factor that characterizes the ALP-mixing with vector resonances. 

In [ ]:
def th_aP(self, mass, coupling, pid0):

    v = 246.22
    cu, cd, cs, cc, cb, ct = 0.873891, 0.986109, 0.986109, 0.873891, 1.04676, 0.906242
    a0, a1, a2, a3 = -25.490, 49.019, -29.482, 5.702
    mPi, mEta, mEtap, fpi, delta = self.masses('111'), self.masses('221'), self.masses('331'), 0.093, 0.370
    F = lambda m: 1 if m <= 1.4 else (a0 + a1*m + a2*m**2 + a3*m**3) if 1.4 < m <=2 else (1.4/m)**4 #if 2 < m <= 3 else 0
    if abs(int(pid0)) == 111: 
        pf = fpi*F(mass)*mass**2*(coupling/(2*v)) / (mass**2 - mPi**2)
        return pf*( cd - cu + (delta*mPi**2/3)*(mass**2*(cd+2*cs+cu)/(mass**2 - mEtap**2) + 2*mass**2*(cu+cd-cs)/(mass**2-mEta**2)))
    elif abs(int(pid0)) == 221: 
        pf = np.sqrt(2/3)*fpi*F(mass)*mass**2*(coupling/(2*v)) / (mass**2 - mEta**2)
        return pf*( cu+cd-cs - delta*mPi**2*(cu-cd)/(mPi**2 - mass**2))
    elif abs(int(pid0)) == 331: 
        pf = np.sqrt(1/3)*fpi*F(mass)*mass**2*(coupling/(2*v)) / (mass**2 - mEtap**2)
        return pf*( -1*(cd+2*cs+cd) - delta*mPi**2*(cu-cd)/(mPi**2 - mass**2))
foresee_module.th_aP = th_aP

In [ ]:
for pid in ['111', '221', '331']: 
    model.add_production_mixing(
        pid = pid,
        mixing = "th_aP(self,mass,coupling,pid0)",
        generator = generators_light,
        energy = energy,
    )

**Decay:** The ALP mainly decays primarily to pairs of fermions, with di-photon decays being relatively sub-dominant. The lifetime and branching fractions for ALPs with universal fermion couplings were computed in [2310.03524](https://arxiv.org/pdf/2310.03524) and are adopted here. 

In [ ]:
model.set_ctau_1d(
    filename="model/ctau.csv", 
    coupling_ref = 2*246.22/1e6
)

decay_modes = ["2e", "2mu", "2tau", "2gamma", "2PichargedPi0", "gamma2Picharged", \
"2Picharged2Pi0", "2Piplus2Piminus", "Eta2Pi0", "Eta2Picharged", \
"2Kstarcharged", "2Kstar0", "Omega2Picharged", "3pi0", "Etapr2Pi0", \
"Etapr2Picharged", "2omega", "Jets-GG", "Jets-cc", "Jets-ss", \
"2KLPi0", "2KSPi0", "KLKSpi0", "2Kstar0", "KminusKLPiplus", \
"KplusKLPiminus", "KminusKSPiplus", "KplusKSPiminus", \
"2Kstarcharged", "2KchargedPi0", "2rho0", "2rhocharged"] 

model.set_br_1d(
    modes = decay_modes,
    finalstates=None,
    filenames=["model/br/"+mode+".csv" for mode in decay_modes],
)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 2. Event Generation

In the following, we want to study one specific benchmark point with $m_{a}=1$ GeV and $g_{aff}= 10^{-3}  \text{ GeV}^{-1}$ and export events as a HEPMC file. 

In [ ]:
mass, coupling, = 1.0, 1e-3

First, we will produce the corresponding flux for this mass and a reference coupling $g_{ref}=1$. 

In [ ]:
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER2 at the HL-LHC. 

In [ ]:
foresee.set_detector(
    distance=650, 
    selection= "-1.5<x.x<1.5 and -.5<x.y<.5" , 
    length=10.0, 
    luminosity=3000, 
)


For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames = ['POWHEG-central', 'POWHEG-max', 'POWHEG-min'][:1]

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=None,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses=[round(x,5) for x in np.logspace(-1,np.log10(6.5),50)]
# Extra points around each production channel kinematic endpoint, from
# utility.production_thresholds(model, mass_range).
thresholds = [
    0.13093, 0.13498, 0.13903, 0.53143, 0.54786, 0.5643, 0.92905, 0.9577,
    0.98651, 5.05873, 5.21519, 5.37164,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-5,np.log10(2e-2),100) 

# Use cached LLP spectra: get_llp_spectrum recomputes on every call,
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function. Below the production rates, we also show the decay branching fractions of the leading visible final states.

In [ ]:
productions=[
     {"channels": ["511","-511","521","-521"]      , "color": "red"    , "label": r"$B  \to X_s a$"  , "generators": generators_heavy},
     {"channels": ["531","-531"]      , "color": "green"  , "label": r"$B_s   \to X_s a$"        , "generators": generators_heavy}, 
     {"channels": ["541","-541"]      , "color": "orange" , "label": r"$B_c \to X_s a$"        , "generators": generators_heavy},    
     {"channels": ["111"]      , "color": "purple"  , "label": r"$\pi^0\to a$"        , "generators": generators_light},
     {"channels": ["221"]      , "color": "brown"  , "label": r"$\eta\to a$"        , "generators": generators_light},
     {"channels": ["331"]      , "color": "cyan"  , "label": r"$\eta\prime\to a$"        , "generators": generators_light},

]

import matplotlib.colors as mcolors

colors= list(mcolors.TABLEAU_COLORS.keys())


branchings = [
    ['2e', colors[0], "solid", r"$e^+e^-$", 0.103, 0.5 ],
    ['2mu', colors[1], "solid", r"$\mu^+\mu^-$", 0.25, 0.5 ],
     ['2tau', colors[2], "solid", r"$\tau^+\tau^-$", 4.7, 0.22 ],
     ['2gamma', colors[3], "solid", r"$\gamma\gamma$", 0.103, 6e-2 ],   
    ['gamma2Picharged', colors[0], "dashed", r"$\gamma2\pi$", 1.2, 1.25e-2 ],   
    ['Eta2Pi0', colors[1], "dashed", r"$\eta2\pi^0$", 1.2, 1.2e-1 ], 
    ['Eta2Picharged', colors[2], "dashed", r"$\eta\pi^+\pi^-$", 1.45, 7e-1 ], 
    ['2omega', colors[3], "dashed", r"$2\omega$", 2.1, 2.6e-2 ],
    ['2rho0', colors[4], "dashed", r"$2\rho^0$", 2.1, 1.5e-2 ],
    ['2rhocharged', colors[5], "dashed", r"$\rho^+\rho^-$", 1.25, .8e-1 ],
    ['Jets-GG', colors[6], "dashed", r"$2G$", 2.4, 4.5e-1 ], 
    ['Jets-cc', colors[7], "dashed", r"$c\bar{c}$", 3.9, .6e-1 ], 
    ['Jets-ss', colors[8], "dashed", r"$s\bar{s}$", 2.1, 1.5e-1 ], 
]
branchingsother = None

plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",  
    xlims=[0.1,10],ylims=[2e-3,4e7],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/g^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(0.9,1),
    fs_label=12,
    ncol=1,
    figsize=(7,6),
    fs_label_br=9,
    branchings=branchings,
    branchingsother=branchingsother,
)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")


Let us now scan over various masses and couplings, and record the resulting number of events. Note that here we again consider the FASER configuration, which we set up before.

In [ ]:
setupnames = ['POWHEG-central']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_POWHEG-central.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_POWHEG-central.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_POWHEG-central.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds, separating the bounds obtained by experimental collaboratios and theory recasts. 

In [ ]:
bounds = [ 
     #["bounds_pi0.txt",       r"$\pi^0\to$ invs.",    1.2e-1, 5e-3, 90      ],
    ["bounds_NA62_A.txt",       r"NA62",    1.1e-1, 2.5e-4, 0       ],
     #["bounds_NA62_B.txt",       None,    2.6e-1, 3.2e-4, 0       ],
    ["bounds_BBN.txt",       "BBN",    1.1e-1, .9e-5, -15       ],
    ["bounds_CHARM_A.txt",       "CHARM",    3e-1, .85e-4, -20       ],
    ["bounds_CHARM_B.txt",       None,    1.12e-1, 1.12e-3, -20      ],
    ["bounds_BK.txt",       r"$B\to K\mu\mu$",    2.95e-1, 6.2e-4, 0       ],
    ["bounds_BKstar.txt",       r"$B\to K^*\mu\mu$",    1.15, 2e-4, 0       ],
   
     
]


We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
    # ["proj_LHCb300.txt", "green",      r"LHCb",    2.8e-1, 2.1e-5, -12],
    #  ["proj_NA62_Kaon.txt", "magenta",      r"NA62",    1.6e-1, 1.e-5, 0       ],
    # ["proj_CODEXb.txt", "purple",      r"CODEXb",   1.7e-0, 1e-5, 0       ],
    #  ["proj_SHiP.txt", "darkblue",      r"SHiP",    2.8e-1, .8e-5, -10       ],
    #["proj_NA62_dump.txt", "green",      r"NA62",    1.1e-1, 2.5e-4, 0       ],
   ]


Finally, we can plot everything using `foresee.plot_reach()`. Here we also add the dark matter relict target line obtained in [2105.07077](https://arxiv.org/abs/2105.07077).

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    projections=projections,
    title="ALPs with universal fermion couplings (BC10)", 
    xlims = [1e-1,2.0], 
    ylims=[8e-6,2e-2],    
    xlabel=r"ALP mass $m_{a}$ [GeV]", 
    ylabel=r"ALP coupling $g_{aff} = 2v/f$ ",
    legendloc=(0.5,.25),
    linewidths=2,
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()